# 05 · Evaluación final
**Proyecto integrador — Lector de Facturas** · Sebastián Rojas y Jesús López

Evalúa el **modelo final «resumen contable»** con la configuración fijada **antes** de leer los documentos (se guardan configuración, hash del código y firma de la muestra):

1. **Test oficial de SROIE y WildReceipt**: 50 documentos por dataset, sin duplicados con desarrollo ni dentro del grupo.
2. **Intervalos de confianza** por bootstrap y **comparación con la línea base** (reglas iniciales y V7 anterior) sobre las mismas lecturas OCR.
3. **Análisis de errores** y dónde funciona bien o mal (tipo de documento, tamaño de imagen, calidad del OCR).
4. **Prueba externa independiente** (opcional): un ZIP de facturas propias anotadas a mano, que nunca se usaron para ajustar reglas.
5. **Lectura de un documento propio** sin anotaciones, para ver qué devuelve el sistema.

> **Honestidad sobre el test.** Durante el desarrollo se revisaron 1.461 documentos (`configs/documentos_revisados.json`), entre ellos casi todo el test oficial. Por eso el punto 1 **no es un test inédito**: sirve para medir y comparar versiones, pero la evidencia de generalización es la prueba externa del punto 4.

**Metas del proyecto:** exactitud de campos ≥ 70 % y documentos procesados sin error ≥ 90 %. Meta interna de mejora: 80 % por campo.

## 1. Configuración

In [ ]:
# ===== Configuración común de los notebooks 01–05 =====
# En Colab no hace falta clonar nada: se descarga el código de GitHub y se registra su commit.
USAR_CODIGO_LOCAL = False  # True si abres Jupyter dentro de un clon local del repositorio
CODIGO_LOCAL = ''          # Opcional: ruta del clon local (por defecto, la carpeta padre de notebooks/)
USAR_DRIVE = False         # True: datos, pesos, caché OCR y resultados quedan en Google Drive (los notebooks no repiten descargas ni OCR)
DESCARGAR_SI_FALTAN = True
DATOS_MANUAL = ''          # Opcional: ruta a data/raw existente en Drive o Colab
REPOSITORIO = 'Jesusl2202/Document-Scanner'
RAMA = 'main'
SEED = 42

from pathlib import Path
import os, sys, json, subprocess, shutil, hashlib, zipfile, io, urllib.request


def comando(args, cwd=None, mostrar_error=True):
    """Ejecuta un comando y muestra la salida solo si falla."""
    r = subprocess.run(list(map(str, args)), cwd=cwd, capture_output=True, text=True)
    if r.returncode:
        if mostrar_error:
            print(r.stdout[-3000:]); print(r.stderr[-3000:])
        r.check_returncode()
    return r.stdout + r.stderr


# 1) Carpeta de trabajo (fuera de Git)
if USAR_CODIGO_LOCAL:
    CODIGO = Path(CODIGO_LOCAL) if CODIGO_LOCAL else (Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd())
    TRABAJO = CODIGO / 'outputs/experimentos'
elif USAR_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    TRABAJO = Path('/content/drive/MyDrive/Lector_Facturas_Experimentos')
else:
    TRABAJO = Path('/content/Lector_Facturas_Experimentos')
TRABAJO.mkdir(parents=True, exist_ok=True)
SALIDAS = TRABAJO / 'notebooks'
SALIDAS.mkdir(parents=True, exist_ok=True)

# 2) Código del repositorio: se descarga el commit exacto y se registra
if USAR_CODIGO_LOCAL:
    REVISION = {'modo': 'local', 'ruta': str(CODIGO)}
else:
    pedido = urllib.request.Request(f'https://api.github.com/repos/{REPOSITORIO}/commits/{RAMA}', headers={'User-Agent': 'lector-facturas'})
    with urllib.request.urlopen(pedido, timeout=60) as respuesta:
        commit = json.load(respuesta)['sha']
    CODIGO = TRABAJO / f'codigo_{commit[:7]}'
    if not (CODIGO / 'src/lector_facturas/consensus.py').exists():
        with urllib.request.urlopen(f'https://codeload.github.com/{REPOSITORIO}/zip/{commit}', timeout=120) as respuesta:
            blob = respuesta.read()
        with zipfile.ZipFile(io.BytesIO(blob)) as z:
            for nombre in z.namelist():
                relativo = '/'.join(nombre.split('/')[1:])
                if not relativo or nombre.endswith('/'):
                    continue
                destino = (CODIGO / relativo).resolve()
                if not destino.is_relative_to(CODIGO.resolve()):
                    raise ValueError('Ruta no permitida dentro del ZIP')
                destino.parent.mkdir(parents=True, exist_ok=True)
                destino.write_bytes(z.read(nombre))
    REVISION = {'repositorio': REPOSITORIO, 'rama': RAMA, 'commit': commit}
assert (CODIGO / 'src/lector_facturas/consensus.py').exists(), 'No se encontró el código del lector.'
RAIZ = CODIGO
print('Código:', REVISION)

# 3) Tesseract y bibliotecas
if shutil.which('apt-get') and not shutil.which('tesseract'):
    print('Instalando Tesseract...')
    comando(['apt-get', 'update', '-qq'])
    comando(['apt-get', 'install', '-y', '-qq', 'tesseract-ocr'])
assert shutil.which('tesseract'), 'No se encontró Tesseract. Usa Google Colab con CPU o instálalo en tu equipo.'
comando([sys.executable, '-m', 'pip', 'install', '-q', '-r', CODIGO / 'requirements.txt'])
sys.path.insert(0, str(CODIGO / 'src'))
for modulo in list(sys.modules):
    if modulo == 'lector_facturas' or modulo.startswith('lector_facturas.'):
        del sys.modules[modulo]

# 4) Modelo: configuración definitiva + pesos Tesseract fijados por SHA-256 (fast, best y español)
from lector_facturas import frozen_runtime
RAW_CONFIG = json.loads((CODIGO / 'configs/definitivo.json').read_text(encoding='utf-8'))
assert hasattr(frozen_runtime, 'bind_models') and RAW_CONFIG.get('observed_accounting'), (
    'El código descargado no corresponde al modelo «resumen contable» (configs/definitivo.json). '
    'Revisa REPOSITORIO y RAMA, o usa USAR_CODIGO_LOCAL=True dentro de un clon del repositorio.')
RAW_CONFIG['seed'] = SEED
CONFIG = frozen_runtime.bind_models(RAW_CONFIG, TRABAJO / 'modelos')
VISTAS_MODELO = len(CONFIG['ocr']['ensemble_views'])
print(comando(['tesseract', '--version']).splitlines()[0], f'| {VISTAS_MODELO} vistas OCR + hasta',
      CONFIG['ocr']['adaptive_rescue']['max_views'], 'de rescate')

# 5) Datos: se buscan por contenido y se descargan si faltan
candidatos = [Path(DATOS_MANUAL)] if DATOS_MANUAL else [
    Path('/content/Document-Scanner/data/raw'), TRABAJO / 'data/raw']
DATOS = next((p for p in candidatos if p.exists()), candidatos[-1])
DATOS.mkdir(parents=True, exist_ok=True)


def localizar():
    """Encuentra la raíz de SROIE y de WildReceipt por su contenido, no por el nombre de la carpeta."""
    sroie = next((p.parent.parent for p in DATOS.rglob('entities') if p.is_dir() and p.parent.name == 'train'
                  and (p.parent / 'img').exists() and (p.parent / 'box').exists()), None)
    wild = next((p.parent for p in DATOS.rglob('class_list.txt') if (p.parent / 'train.txt').exists() and (p.parent / 'test.txt').exists()), None)
    return sroie, wild


SROIE, WILD = localizar()
for nombre, ruta in [('sroie', SROIE), ('wildreceipt', WILD)]:
    if ruta is None and DESCARGAR_SI_FALTAN:
        print('Descargando', nombre, '...', flush=True)
        try:
            print(comando([sys.executable, CODIGO / 'scripts/download_data.py', nombre, '--out', DATOS], mostrar_error=False))
        except subprocess.CalledProcessError:
            print(f'No se pudo descargar {nombre}. Descárgalo a mano en {DATOS} o indica DATOS_MANUAL (ver data/raw/README.md).')
SROIE, WILD = localizar()
print('SROIE:      ', SROIE or 'NO ENCONTRADO')
print('WildReceipt:', WILD or 'NO ENCONTRADO')
assert SROIE and WILD, 'Faltan datasets: revisa data/raw/README.md.'

# 6) Manifiesto con la partición del proyecto (train / validation / test / quarantine)
from lector_facturas.datasets import prepare
FILAS = prepare(SROIE, WILD, SALIDAS / 'preparados', RAW_CONFIG)
REVISADOS = json.loads((CODIGO / 'configs/documentos_revisados.json').read_text(encoding='utf-8'))
print('Documentos en el manifiesto:', len(FILAS), '| documentos ya revisados en el desarrollo:', len(REVISADOS))

### 1.1 Librerías, parámetros y funciones
Ejecuta esta celda antes de continuar.

In [ ]:
# Funciones de gráficos compartidas por los notebooks 01–05 (se repiten para que cada uno funcione solo en Colab).
import glob, re, json, types
from collections import Counter
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

AZUL, NARANJA, AQUA, GRIS = '#2a78d6', '#eb6834', '#1baf7a', '#8a8984'

FONDO, TEXTO, TEXTO2, REJILLA = '#fcfcfb', '#0b0b0b', '#52514e', '#e4e3df'

COLOR = {'sroie': AZUL, 'wildreceipt': NARANJA}

NOMBRE = {'sroie': 'SROIE', 'wildreceipt': 'WildReceipt'}

CAMPOS = ['tax_id', 'date', 'document_number', 'subtotal', 'tax', 'total']

plt.rcParams.update({
    'figure.facecolor': FONDO, 'axes.facecolor': FONDO, 'axes.edgecolor': REJILLA,
    'axes.labelcolor': TEXTO2, 'xtick.color': TEXTO2, 'ytick.color': TEXTO2, 'text.color': TEXTO,
    'font.size': 10, 'axes.titlesize': 12, 'axes.titleweight': 'bold',
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True,
    'grid.color': REJILLA, 'grid.linewidth': .6, 'axes.axisbelow': True, 'legend.frameon': False})

def es_num(valor, decimales=0):
    """Formato numérico en español: 1.234,5."""
    texto = f'{valor:,.{decimales}f}'
    return texto.replace(',', 'X').replace('.', ',').replace('X', '.')

def guardar(fig, carpeta, nombre):
    fig.tight_layout()
    fig.savefig(Path(carpeta) / nombre, dpi=150)
    plt.close(fig)
    print('  figura:', nombre)

def leer_csv(ruta):
    return pd.read_csv(ruta, encoding='utf-8-sig')

def fig_particiones(documentos, incidencias, manifest, salida):
    """Figura 1. Documentos por partición y dataset."""
    if manifest is not None:
        conteo = manifest.groupby(['dataset', 'split']).size().unstack(fill_value=0)
    else:
        # Sin manifiesto solo se conocen desarrollo y cuarentena.
        conteo = documentos.groupby(['dataset', 'split']).size().unstack(fill_value=0)
        cuarentena = incidencias[incidencias.split == 'quarantine'].drop_duplicates('id')
        conteo['quarantine'] = cuarentena.groupby('dataset').size().reindex(conteo.index, fill_value=0)
        print('  aviso: sin --manifest la Figura 1 no muestra el test oficial')
    columnas = {'train': ('Entrenamiento', AZUL), 'validation': ('Validación', AQUA),
                'test': ('Test oficial (reservado)', GRIS), 'quarantine': ('Cuarentena', NARANJA)}
    total = int(conteo.values.sum())
    fig, ax = plt.subplots(figsize=(8, 3.2))
    izquierda = np.zeros(len(conteo))
    etiquetas = [NOMBRE.get(d, d) for d in conteo.index]
    for clave, (texto, color) in columnas.items():
        if clave not in conteo:
            continue
        valores = conteo[clave].values
        ax.barh(etiquetas, valores, left=izquierda, color=color, label=texto,
                edgecolor=FONDO, linewidth=2, height=.55)
        for i, v in enumerate(valores):
            if v > 60:
                ax.text(izquierda[i] + v / 2, i, es_num(v), ha='center', va='center',
                        color='white', fontsize=9, fontweight='bold')
        izquierda += valores
    ax.set_xlabel('Documentos')
    ax.set_title(f'Figura 1. Partición de los {es_num(total)} documentos', loc='left')
    ax.legend(ncol=4, loc='upper center', bbox_to_anchor=(.5, -.28))
    ax.grid(axis='y', visible=False)
    ax.invert_yaxis()
    guardar(fig, salida, '01_particiones.png')
    return conteo

def fig_resolucion(documentos, salida):
    """Figura 2. Ancho vs. alto de las imágenes de desarrollo."""
    fig, ax = plt.subplots(figsize=(7, 4.6))
    for ds in ['wildreceipt', 'sroie']:
        g = documentos[documentos.dataset == ds]
        if g.empty:
            continue
        ax.scatter(g.width, g.height, s=12, alpha=.45, color=COLOR[ds], edgecolors='none',
                   label=f'{NOMBRE[ds]} (mediana {int(g.width.median())}×{int(g.height.median())} px)')
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel('Ancho (px, escala log)')
    ax.set_ylabel('Alto (px, escala log)')
    ax.set_title('Figura 2. Resolución de las imágenes de desarrollo', loc='left')
    ax.legend(loc='upper left')
    guardar(fig, salida, '02_resolucion.png')

def fig_regiones(documentos, salida):
    """Figura 3. Regiones de texto anotadas por documento."""
    fig, ax = plt.subplots(figsize=(7.5, 3.8))
    bins = np.arange(0, documentos.regions.max() + 6, 6)
    for ds in ['sroie', 'wildreceipt']:
        g = documentos[documentos.dataset == ds].regions
        if g.empty:
            continue
        ax.hist(g, bins=bins, alpha=.6, color=COLOR[ds], edgecolor=FONDO, linewidth=1,
                label=f'{NOMBRE[ds]} (mediana {int(g.median())})')
        ax.axvline(g.median(), color=COLOR[ds], ls='--', lw=1.5)
    ax.set_xlabel('Regiones de texto anotadas por documento')
    ax.set_ylabel('Documentos')
    ax.set_title('Figura 3. Cantidad de texto por recibo', loc='left')
    ax.legend()
    guardar(fig, salida, '03_regiones.png')

def fig_cobertura(cobertura, salida):
    """Figura 4. Porcentaje de documentos con etiqueta utilizable por campo."""
    tabla = cobertura.pivot(index='dataset', columns='field', values='coverage').reindex(columns=CAMPOS)
    mapa = LinearSegmentedColormap.from_list('azules', ['#f1f0ec', '#cfe0f6', '#7fb0ea', AZUL, '#0d3f80'])
    fig, ax = plt.subplots(figsize=(8, 1.3 + .65 * len(tabla)))
    ax.imshow(tabla.values, cmap=mapa, vmin=0, vmax=1, aspect='auto')
    ax.grid(False)
    ax.set_xticks(range(len(CAMPOS)), CAMPOS)
    ax.set_yticks(range(len(tabla)), [NOMBRE.get(d, d) for d in tabla.index])
    for i in range(tabla.shape[0]):
        for j in range(tabla.shape[1]):
            v = tabla.values[i, j]
            ax.text(j, i, f'{es_num(100 * v)}%', ha='center', va='center',
                    color='white' if v > .55 else TEXTO, fontweight='bold')
    for borde in ax.spines.values():
        borde.set_visible(False)
    ax.set_title('Figura 4. Campos con etiqueta utilizable (documentos de desarrollo)', loc='left')
    guardar(fig, salida, '04_cobertura.png')

def fig_clases_lineas(texto, salida):
    """Figura 5. Desbalance de clases por línea (datos del clasificador de Semana 3)."""
    conteo = pd.Series({k.strip(): int(v) for k, v in (p.split('=') for p in texto.split(','))}).sort_values()
    total = conteo.sum()
    fig, ax = plt.subplots(figsize=(7.5, 3.2))
    ax.barh(conteo.index, conteo.values, height=.55,
            color=[GRIS if k.lower().startswith('otro') else AZUL for k in conteo.index])
    for i, v in enumerate(conteo.values):
        ax.text(v * 1.08, i, f'{es_num(v)} ({es_num(100 * v / total, 1)}%)', va='center', color=TEXTO2, fontsize=9)
    ax.set_xscale('log')
    ax.set_xlim(conteo.min() / 2, conteo.max() * 4)
    ax.set_xlabel('Líneas OCR en entrenamiento (escala log)')
    ax.grid(axis='y', visible=False)
    ax.set_title('Figura 5. Desbalance de clases por línea', loc='left')
    guardar(fig, salida, '05_desbalance_lineas.png')

def fig_cer(cer, documentos, salida):
    """Figura 6. CER del OCR frente al tamaño de la imagen."""
    datos = cer.merge(documentos, on=['id', 'dataset'])
    if datos.empty:
        print('  aviso: los documentos de cer.csv no están en documents.csv; se omite la Figura 6')
        return None
    datos['mpx'] = datos.width * datos.height / 1e6
    rho = datos[['cer', 'mpx']].corr(method='spearman').iloc[0, 1]
    fig, ax = plt.subplots(figsize=(7, 4.2))
    for ds in ['sroie', 'wildreceipt']:
        g = datos[datos.dataset == ds]
        if g.empty:
            continue
        ax.scatter(g.mpx, g.cer, s=46, color=COLOR[ds], edgecolors=FONDO, linewidths=1.5,
                   label=f'{NOMBRE[ds]} (CER mediano {es_num(g.cer.median(), 2)})')
    ax.set_xscale('log')
    ax.set_xlabel('Tamaño de la imagen (megapíxeles, escala log)')
    ax.set_ylabel('CER del OCR')
    ax.set_title(f'Figura 6. Imágenes más pequeñas se leen peor (Spearman ρ = {es_num(rho, 2)})', loc='left')
    ax.legend()
    guardar(fig, salida, '06_cer_vs_resolucion.png')
    return datos

def tipo_error(errores):
    """Clasifica cada campo evaluado en correcto, omitido o valor incorrecto."""
    vacio = errores.predicted.isna() | (errores.predicted.astype(str).str.strip() == '')
    return np.where(errores.correct == 1, 'Correcto',
                    np.where(vacio, 'Omitido (no extrajo nada)', 'Valor incorrecto'))

def fig_errores(errores, salida):
    """Figura 7. Resultado por campo: correcto, valor incorrecto u omitido."""
    errores = errores.assign(tipo=tipo_error(errores))
    orden = ['Correcto', 'Valor incorrecto', 'Omitido (no extrajo nada)']
    tabla = errores.groupby(['dataset', 'field', 'tipo']).size().unstack(fill_value=0).reindex(columns=orden, fill_value=0)
    tabla.index = [f'{NOMBRE.get(d, d)} · {c}' for d, c in tabla.index]
    documentos = errores.id.nunique()
    fig, ax = plt.subplots(figsize=(8, 1.4 + .4 * len(tabla)))
    izquierda = np.zeros(len(tabla))
    for col, color in zip(orden, [AQUA, NARANJA, '#b9b8b2']):
        valores = tabla[col].values
        ax.barh(tabla.index, valores, left=izquierda, color=color, label=col, edgecolor=FONDO, linewidth=2, height=.6)
        for i, v in enumerate(valores):
            if v >= 2:
                ax.text(izquierda[i] + v / 2, i, str(v), ha='center', va='center', fontsize=9,
                        color=TEXTO if col.startswith('Omitido') else 'white')
        izquierda += valores
    ax.invert_yaxis()
    ax.set_xlabel('Campos anotados')
    ax.grid(axis='y', visible=False)
    ax.set_title(f'Figura 7. Resultado por campo ({documentos} documentos)', loc='left')
    ax.legend(ncol=3, loc='upper center', bbox_to_anchor=(.5, -.2))
    guardar(fig, salida, '07_errores_linea_base.png')
    return errores

def fig_correlaciones(documentos, salida):
    """Figura 8. Correlación de Spearman entre variables de la imagen, por dataset."""
    variables = {'width': 'Ancho', 'height': 'Alto', 'mpx': 'Megapíxeles', 'regions': 'Regiones', 'aspect': 'Alto/ancho'}
    mapa = LinearSegmentedColormap.from_list('div', [NARANJA, '#f6c3ad', '#f1f0ec', '#a9c8f0', AZUL])
    bases = [d for d in ['sroie', 'wildreceipt'] if d in set(documentos.dataset)]
    fig, ejes = plt.subplots(1, len(bases), figsize=(4.5 * len(bases), 3.8), squeeze=False)
    matrices = {}
    for ax, ds in zip(ejes[0], bases):
        m = documentos[documentos.dataset == ds][list(variables)].corr(method='spearman')
        matrices[ds] = m
        ax.imshow(m.values, cmap=mapa, vmin=-1, vmax=1)
        ax.grid(False)
        ax.set_xticks(range(len(variables)), variables.values(), rotation=35, ha='right')
        ax.set_yticks(range(len(variables)), variables.values())
        for i in range(len(variables)):
            for j in range(len(variables)):
                v = m.values[i, j]
                ax.text(j, i, es_num(v, 2), ha='center', va='center', fontsize=8,
                        color='white' if abs(v) > .6 else TEXTO)
        ax.set_title(NOMBRE[ds])
        for borde in ax.spines.values():
            borde.set_visible(False)
    fig.suptitle('Figura 8. Correlación de Spearman entre variables de la imagen', x=.02, ha='left', fontweight='bold')
    guardar(fig, salida, '08_correlaciones.png')
    return matrices

PATRONES_FECHA = {'99/99/9999': 'dd/mm/aaaa', '99-99-99': 'dd-mm-aa', '99-99-9999': 'dd-mm-aaaa',
                  '99/99/99': 'dd/mm/aa', '99 A 9999': 'dd MES aaaa'}

def patron(fecha):
    """Convierte '25/12/2018' en '99/99/9999' y 'MAR' en 'A' para agrupar formatos."""
    return re.sub(r'\d', '9', re.sub(r'[A-Za-z]+', 'A', fecha))

def fig_formatos_fecha(carpeta, salida):
    """Figura 9. Formatos de fecha en las etiquetas de entrenamiento de SROIE."""
    fechas = []
    for archivo in sorted(glob.glob(str(Path(carpeta) / '*'))):
        if not archivo.endswith(('.json', '.txt')):
            continue
        try:
            fechas.append(str(json.loads(Path(archivo).read_text(encoding='utf-8-sig')).get('date', '')).strip())
        except (json.JSONDecodeError, UnicodeDecodeError):
            print('  aviso: no se pudo leer', archivo)
    if not fechas:
        print('  aviso: no hay etiquetas en --sroie-keys; se omite la Figura 9')
        return None
    formatos = Counter(patron(f) for f in fechas)
    ambiguas = 0
    for f in fechas:
        m = re.fullmatch(r'\(?(\d{1,2})[/.\-](\d{1,2})[/.\-](\d{2,4})\)?', f)
        if m and int(m.group(1)) <= 12 and int(m.group(2)) <= 12 and m.group(1) != m.group(2):
            ambiguas += 1
    agrupado = Counter()
    for p, n in formatos.items():
        agrupado[PATRONES_FECHA.get(p, 'otros')] += n
    otros = len([p for p in formatos if p not in PATRONES_FECHA])
    orden = [v for v in PATRONES_FECHA.values() if agrupado[v]] + (['otros'] if agrupado['otros'] else [])
    orden = orden[::-1]
    etiquetas = [f'Otros {otros} formatos' if o == 'otros' else o for o in orden]
    valores = [agrupado[o] for o in orden]
    n = len(fechas)
    fig, ax = plt.subplots(figsize=(7.5, 3.4))
    ax.barh(etiquetas, valores, color=AZUL, height=.55)
    ax.grid(axis='y', visible=False)
    for i, v in enumerate(valores):
        ax.text(v + n * .006, i, f'{v} ({es_num(100 * v / n, 1)}%)', va='center', color=TEXTO2, fontsize=9)
    ax.set_xlim(0, max(valores) * 1.2)
    ax.set_xlabel(f'Recibos de entrenamiento de SROIE (n = {n})')
    ax.set_title('Figura 9. Formatos de fecha en las etiquetas de SROIE', loc='left')
    guardar(fig, salida, '09_formatos_fecha.png')
    return {'recibos': n, 'formatos_distintos': len(formatos), 'fechas_dia_mes_ambiguas': ambiguas,
            'porcentaje_ambiguas': round(100 * ambiguas / n, 1),
            'formatos': dict(formatos.most_common())}

def tablas(documentos, salida):
    """Estadísticas descriptivas y outliers (criterio 1,5 × IQR) por dataset."""
    variables = ['width', 'height', 'aspect', 'mpx', 'regions']
    describe = documentos.groupby('dataset')[variables].describe(percentiles=[.25, .5, .75]).T.round(2)
    describe.to_csv(Path(salida) / 'estadisticas_descriptivas.csv', encoding='utf-8-sig')
    filas = []
    for ds, g in documentos.groupby('dataset'):
        for v in ['width', 'height', 'mpx', 'regions']:
            q1, q3 = g[v].quantile([.25, .75])
            limite_bajo, limite_alto = q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1)
            n = int(((g[v] < limite_bajo) | (g[v] > limite_alto)).sum())
            filas.append({'dataset': ds, 'variable': v, 'outliers': n, 'documentos': len(g),
                          'porcentaje': round(100 * n / len(g), 1),
                          'limite_inferior': round(limite_bajo, 2), 'limite_superior': round(limite_alto, 2)})
    outliers = pd.DataFrame(filas)
    outliers.to_csv(Path(salida) / 'outliers_iqr.csv', index=False, encoding='utf-8-sig')
    print('  tablas: estadisticas_descriptivas.csv, outliers_iqr.csv')
    return describe, outliers

F = types.SimpleNamespace(AZUL=AZUL, NARANJA=NARANJA, AQUA=AQUA, GRIS=GRIS, FONDO=FONDO, TEXTO=TEXTO, TEXTO2=TEXTO2, REJILLA=REJILLA, COLOR=COLOR, NOMBRE=NOMBRE, CAMPOS=CAMPOS, es_num=es_num, guardar=guardar, leer_csv=leer_csv, fig_particiones=fig_particiones, fig_resolucion=fig_resolucion, fig_regiones=fig_regiones, fig_cobertura=fig_cobertura, fig_clases_lineas=fig_clases_lineas, fig_cer=fig_cer, tipo_error=tipo_error, fig_errores=fig_errores, fig_correlaciones=fig_correlaciones, PATRONES_FECHA=PATRONES_FECHA, patron=patron, fig_formatos_fecha=fig_formatos_fecha, tablas=tablas)

In [ ]:
import json, re, hashlib, warnings, copy, time, math
from collections import Counter
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image as Img, Markdown
warnings.filterwarnings('ignore', category=FutureWarning)
%matplotlib inline
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 160)
from lector_facturas import consensus, candidate_rules
from lector_facturas.common import FIELDS, normalize, write_json, write_jsonl
from lector_facturas.ocr import transcribe, render_pdf
from lector_facturas.evaluation import evaluate
from lector_facturas.experiments import run, metrics, bootstrap_delta, random_known_sample, load_external_test
from lector_facturas.frozen_runtime import bind_models, review_notes, extraction_status
from lector_facturas.cli import predict
N_POR_BASE = 50                 # documentos del test oficial por dataset
SEMILLA_TEST = SEED             # None = un grupo aleatorio distinto en cada ejecución
USAR_CONFIG_OPTIMIZADA = False  # True: aplica configuracion_recomendada.json del notebook 04
RUTA_CONFIG_OPTIMIZADA = ''     # Vacío: se busca en la carpeta de salidas de 04 o se pide subirla
EXTERNO_ZIP = ''                # Opcional: ZIP con anotaciones.jsonl + imagenes/ (formato en docs/anotacion_propia.md)
CACHE = SALIDAS / 'cache_ocr_test'
EVAL = SALIDAS / ('evaluacion_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')); EVAL.mkdir(parents=True)
PRIV = EVAL / 'documento_privado'   # imágenes y lecturas propias: nunca entran al ZIP
FIG = EVAL / 'figuras'; FIG.mkdir()

## 2. Configuración final (opcional: la recomendada por 04)
Si `USAR_CONFIG_OPTIMIZADA = True`, se aplica `configuracion_recomendada.json` exportada por el notebook 04 **antes** de elegir la muestra. Solo se aceptan los hiperparámetros que 04 optimiza.

In [ ]:
PARAMS_CONSENSO = {'primary_weight', 'vote_weight', 'fuzzy_weight', 'balance_bonus'}
ETAPAS_PERMITIDAS = {'printed_improvements', 'semantic_amounts', 'document_consistency', 'structured_fields',
                     'observed_accounting', 'confidence_rescue'}
VISTAS_PERMITIDAS = {'usar_pesos_best', 'usar_vista_espanol', 'usar_rescate_ocr'}


def aplicar_recomendacion(raw, recomendada):
    """Aplica la salida de 04 sobre la configuración cruda, validando cada valor."""
    cfg = copy.deepcopy(raw)
    hp = recomendada.get('consensus_parameters', {})
    if not set(hp) <= PARAMS_CONSENSO or any(not isinstance(v, (int, float)) or not 0 <= v <= 6 for v in hp.values()):
        raise ValueError('Parámetros de consenso no válidos')
    etapas, vistas = recomendada.get('etapas', {}), recomendada.get('vistas', {})
    if not set(etapas) <= ETAPAS_PERMITIDAS or not set(vistas) <= VISTAS_PERMITIDAS:
        raise ValueError('La recomendación contiene claves desconocidas')
    if any(not isinstance(v, bool) for v in list(etapas.values()) + list(vistas.values())):
        raise ValueError('Etapas y vistas deben ser verdadero/falso')
    cfg['consensus_parameters'] = hp
    cfg.update(etapas)
    if vistas.get('usar_pesos_best') is False:
        cfg['ocr']['ensemble_views'] = [v for v in cfg['ocr']['ensemble_views'] if v.get('model') != 'best']
    if vistas.get('usar_pesos_best') is False or vistas.get('usar_rescate_ocr') is False:
        cfg['ocr']['adaptive_rescue']['enabled'] = False
        cfg['ocr']['summary_rescue']['enabled'] = False
    if vistas.get('usar_vista_espanol') is False:
        cfg['ocr']['include_spanish'] = False
    return cfg


RECOMENDADA = None
if USAR_CONFIG_OPTIMIZADA:
    ruta = Path(RUTA_CONFIG_OPTIMIZADA) if RUTA_CONFIG_OPTIMIZADA else SALIDAS / 'optimizacion/configuracion_recomendada.json'
    if ruta.exists():
        RECOMENDADA = json.loads(ruta.read_text(encoding='utf-8'))
    else:
        from google.colab import files
        print('Sube configuracion_recomendada.json exportada por 04_optimizacion.')
        subido = files.upload()
        RECOMENDADA = json.loads(next(iter(subido.values())).decode('utf-8'))
    RAW_FINAL = aplicar_recomendacion(RAW_CONFIG, RECOMENDADA)
else:
    RAW_FINAL = copy.deepcopy(RAW_CONFIG)
CONFIG_FINAL = bind_models(RAW_FINAL, TRABAJO / 'modelos')
print('Configuración:', 'recomendada por 04 (' + RECOMENDADA.get('decision', '?') + ')' if RECOMENDADA else 'entregada (configs/definitivo.json)',
      '|', len(CONFIG_FINAL['ocr']['ensemble_views']), 'vistas OCR')

## 3. Congelar configuración y muestra (antes de leer el test)
`experiments.random_known_sample`: documentos del test oficial cuya imagen no aparece en entrenamiento, sin repetir imágenes. Con `SEMILLA_TEST = SEED` la muestra es reproducible; con `None` cada ejecución toma un grupo nuevo.

In [ ]:
sample, plan = random_known_sample(FILAS, N_POR_BASE, seed=SEMILLA_TEST)
assert all(r['split'] == 'test' and r['source_split'] == 'test' for r in sample)
revisados = set(REVISADOS) | set(REVISADOS.values())
plan.update({'revision_codigo': REVISION, 'configuracion': RAW_FINAL, 'recomendacion_04': RECOMENDADA,
             'codigo_sha256': {str(p.relative_to(CODIGO)): hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted((CODIGO / 'src').rglob('*.py'))},
             'firma_muestra': hashlib.sha256(json.dumps(sorted((r['id'], r['sha256']) for r in sample)).encode()).hexdigest(),
             'ya_revisados_en_desarrollo': sum(r['id'] in revisados or r['sha256'] in revisados for r in sample),
             'nota': 'Configuración fija antes de leer el test; no ajustar con estos resultados.'})
write_json(EVAL / 'plan_test.json', plan)
write_jsonl(EVAL / 'muestra_test_congelada.jsonl', sample)
soporte = pd.DataFrame([{'dataset': r['dataset'], 'campo': f} for r in sample for f in FIELDS if r['gold'][f]['status'] == 'annotated'])
print('Firma de la muestra:', plan['firma_muestra'][:16], '... | documentos ya revisados en el desarrollo:', plan['ya_revisados_en_desarrollo'], 'de', len(sample))
display(soporte.value_counts().rename('campos con referencia').reset_index().pivot(index='campo', columns='dataset', values='campos con referencia').fillna(0).astype(int))

## 4. Ejecución del modelo final sobre el test
`experiments.run`: OCR de todas las vistas + extractor + validación + `evaluate`. Puede tardar 30–90 minutos en Colab CPU (100 documentos × 10 vistas).

In [ ]:
inicio = time.perf_counter()
FINAL = run(sample, CONFIG_FINAL, EVAL / 'test_final', CACHE)
print(f'Duración: {(time.perf_counter() - inicio) / 60:.1f} min')
resumen = json.loads((FINAL / 'evaluation/summary.json').read_text())
cumplimiento = {'exactitud': resumen['micro_exact_match'], 'procesamiento': resumen['execution_success_rate'],
                'cumple_exactitud_70': resumen['micro_exact_match'] >= .70, 'cumple_procesamiento_90': resumen['execution_success_rate'] >= .90,
                'documentos': resumen['documents'], 'campos_evaluados': resumen['evaluated_field_instances'],
                'mediana_s_por_documento': resumen['median_seconds_uncached'], 'p95_s_por_documento': resumen['p95_seconds_uncached'],
                'cer': resumen['cer_layout_order_sensitive']}
write_json(EVAL / 'cumplimiento_meta.json', cumplimiento)
display(pd.DataFrame([cumplimiento]).T.rename(columns={0: 'valor'}))
print('Metas cumplidas en esta muestra.' if cumplimiento['cumple_exactitud_70'] and cumplimiento['cumple_procesamiento_90']
      else 'Alguna meta no se alcanzó; se reporta el resultado sin cambiar la configuración.')

In [ ]:
metricas = metrics(FINAL).query('support > 0').copy()
metricas['meta_80'] = metricas.exact_match >= .80
metricas['faltan_para_80'] = (metricas.support.mul(.8).apply(math.ceil) - metricas.tp).clip(lower=0).astype(int)
tabla_metricas = metricas.assign(exact_match=(100 * metricas.exact_match).round(1), precision=metricas.precision.round(3),
                                 recall=metricas.recall.round(3), f1=metricas.f1.round(3))
display(tabla_metricas[['dataset', 'field', 'support', 'exact_match', 'precision', 'recall', 'f1', 'tp', 'fp', 'fn', 'meta_80', 'faltan_para_80']])
tabla_metricas.to_csv(EVAL / 'metricas_por_campo.csv', index=False, encoding='utf-8-sig')

## 5. Intervalos de confianza
Bootstrap por documento (2.000 remuestreos, estratificado por dataset) para la exactitud micro.

In [ ]:
errores = pd.read_csv(FINAL / 'evaluation/errors.csv', encoding='utf-8-sig')
por_doc = errores.groupby(['dataset', 'id']).agg(ok=('correct', 'sum'), n=('correct', 'size')).reset_index()
rng = np.random.default_rng(SEED)

def ic(df, reps=2000):
    grupos = [g for _, g in df.groupby('dataset')]
    vals = []
    for _ in range(reps):
        m = pd.concat([g.iloc[rng.integers(0, len(g), len(g))] for g in grupos])
        vals.append(m.ok.sum() / m.n.sum())
    return np.percentile(vals, [2.5, 97.5])

filas_ic = [{'grupo': 'todos', 'exactitud': por_doc.ok.sum() / por_doc.n.sum(), **dict(zip(['ic95_inf', 'ic95_sup'], ic(por_doc)))}]
for base, g in por_doc.groupby('dataset'):
    filas_ic.append({'grupo': base, 'exactitud': g.ok.sum() / g.n.sum(), **dict(zip(['ic95_inf', 'ic95_sup'], ic(g)))})
tabla_ic = pd.DataFrame(filas_ic).set_index('grupo').mul(100).round(1)
tabla_ic.to_csv(EVAL / 'intervalos_confianza.csv', encoding='utf-8-sig')
display(tabla_ic)

## 6. Comparación con la línea base
Con las **mismas lecturas OCR** del test (salen de la caché) se calculan:
- **M0 · Reglas iniciales**: solo las reglas V5 sobre la vista principal (una lectura).
- **M1 · V7 anterior**: consenso de las 5 vistas `fast` del modelo anterior, sin las etapas nuevas.
- **M2 · Actual sin resumen contable**: todas las vistas y etapas, salvo contabilidad observada y rescate por confianza.
- **M3 · Modelo final**: el resultado de la sección 4.

In [ ]:
ETAPAS = ['printed_improvements', 'accounting_improvements', 'amount_context', 'spatial_amounts', 'semantic_amounts',
          'document_consistency', 'structured_fields', 'observed_accounting', 'confidence_rescue']


def es_vista_v7(vista):
    o = vista['options']
    return not vista.get('rescue') and o.get('language', 'eng') == 'eng' and Path(o.get('tessdata_dir', '')).name == 'fast'


def solo_vistas(o, condicion):
    elegidas = [v for v in o['views'] if condicion(v)]
    principal = next((v for v in elegidas if consensus.is_primary(v)), elegidas[0])
    return dict(principal, views=elegidas)


def vista_principal(o):
    principal = next(v for v in o['views'] if consensus.is_primary(v) and es_vista_v7(v))
    return dict(principal, views=[principal])


lecturas = {}
for r in sample:
    try:
        lecturas[r['id']] = transcribe(r['image'], CONFIG_FINAL['ocr'], CACHE)
    except Exception as exc:
        print('Sin OCR para', r['id'], exc)


def evaluar_modelo(nombre, extractor):
    preds = []
    for r in sample:
        try:
            out = extractor(lecturas[r['id']])
            preds.append({'id': r['id'], 'dataset': r['dataset'], 'fields': out['fields'], 'status': 'ok', 'text': '', 'elapsed_seconds': None, 'cache_hit': True})
        except Exception as exc:
            preds.append({'id': r['id'], 'dataset': r['dataset'], 'fields': {}, 'status': 'error', 'error': str(exc), 'text': ''})
    carpeta = EVAL / nombre
    return evaluate(sample, preds, CONFIG_FINAL, carpeta / 'evaluation', calculate_cer=False), carpeta


modelos = {'M0 reglas iniciales (1 vista)': lambda o: candidate_rules.extract(vista_principal(o), CONFIG_FINAL),
           'M1 V7 anterior (5 vistas fast)': lambda o: consensus.extract(solo_vistas(o, es_vista_v7), dict(CONFIG_FINAL, **{e: False for e in ETAPAS})),
           'M2 actual sin resumen contable': lambda o: consensus.extract(o, dict(CONFIG_FINAL, observed_accounting=False, confidence_rescue=False))}
resultados = {n: evaluar_modelo(n.split()[0], f) for n, f in modelos.items()}
resultados['M3 resumen contable (final)'] = (resumen, FINAL)
comparacion = pd.DataFrame({n: {'exactitud %': 100 * s['micro_exact_match'], 'procesados %': 100 * s['execution_success_rate'],
                                'campos': s['evaluated_field_instances']} for n, (s, _) in resultados.items()}).T.round(1)
comparacion.to_csv(EVAL / 'comparacion_modelos.csv', encoding='utf-8-sig')
display(comparacion)
nombres_m = list(resultados)
deltas = pd.DataFrame([dict(comparacion=f'{a} → {nombres_m[-1]}', **bootstrap_delta(resultados[a][1], resultados[nombres_m[-1]][1]))
                       for a in nombres_m[:-1]]).round(2)
deltas.to_csv(EVAL / 'deltas_bootstrap.csv', index=False, encoding='utf-8-sig')
display(deltas)

In [ ]:
tabla_campos = []
for n, (s, carpeta) in resultados.items():
    m = metrics(carpeta).query("support > 0 and dataset == 'ALL'")
    tabla_campos += [{'modelo': n, 'campo': r.field, 'exactitud': 100 * r.exact_match} for r in m.itertuples()]
tabla_campos = pd.DataFrame(tabla_campos)
campos = list(dict.fromkeys(tabla_campos.campo))
fig, ax = plt.subplots(figsize=(9, 4))
ancho = .8 / len(resultados)
for i, (n, color) in enumerate(zip(resultados, [F.GRIS, F.NARANJA, F.AQUA, F.AZUL])):
    v = [tabla_campos[(tabla_campos.modelo == n) & (tabla_campos.campo == c)].exactitud.mean() for c in campos]
    ax.bar(np.arange(len(campos)) + i * ancho, v, ancho, label=n, color=color)
ax.axhline(70, color='#52514e', ls='--', lw=1); ax.text(len(campos) - .6, 71, 'meta 70 %', fontsize=8)
ax.set_xticks(np.arange(len(campos)) + ancho * (len(resultados) - 1) / 2, campos); ax.set_ylim(0, 100)
ax.set_ylabel('Exactitud (%)'); ax.set_title('Test oficial: exactitud por campo y modelo', loc='left'); ax.legend(fontsize=8)
plt.tight_layout(); fig.savefig(FIG / 'test_comparacion_modelos.png', dpi=150); plt.show()

### 6.1 Referencias externas
Las cifras de la literatura no son directamente comparables (otras métricas, otros subconjuntos, OCR de referencia en lugar de OCR propio y modelos entrenados con esos datos), pero sirven para ubicar el resultado.

In [ ]:
display(pd.DataFrame([
    {'referencia': 'Meta mínima del proyecto (Documento de Alcance)', 'valor': '≥ 70 % exactitud por campo'},
    {'referencia': 'Meta interna de mejora', 'valor': '≥ 80 % por campo'},
    {'referencia': 'Sistemas del reto SROIE e información clave en WildReceipt (Huang et al., 2019; Sun et al., 2021)',
     'valor': 'F1 > 85–90 %, con modelos entrenados en esos datos'},
    {'referencia': 'Este proyecto (modelo final, test oficial)',
     'valor': f"{100 * resumen['micro_exact_match']:.1f} % [IC95 {tabla_ic.loc['todos', 'ic95_inf']:.1f}–{tabla_ic.loc['todos', 'ic95_sup']:.1f}]"}]))

## 7. Análisis de errores

In [ ]:
errores = errores.assign(tipo=F.tipo_error(errores))
display(pd.crosstab([errores.dataset, errores.field], errores.tipo, margins=True))
F.fig_errores(errores, FIG)
display(Img(filename=str(FIG / '07_errores_linea_base.png')))
display(Markdown('**Ejemplos de valores incorrectos** (datasets públicos)'))
display(errores[errores.tipo == 'Valor incorrecto'][['id', 'field', 'gold', 'predicted', 'gold_normalized', 'predicted_normalized']].head(15))

In [ ]:
predicciones = [json.loads(l) for l in (FINAL / 'predictions.jsonl').read_text(encoding='utf-8').splitlines() if l.strip()]
alertas = Counter(a for p in predicciones for a in p.get('alerts', []))
display(pd.Series(alertas, dtype=int).sort_values(ascending=False).rename('documentos').to_frame())
# ¿Qué tan confiables son los campos recuperados por el rescate o corregidos por el resumen contable?
for sufijo in ('confidence_gap_recovery', 'observed_accounting_selection'):
    ids_campo = {(p['id'], a.split(':')[0]) for p in predicciones for a in p.get('alerts', []) if a.endswith(sufijo)}
    t = errores[[(i, f) in ids_campo for i, f in zip(errores.id, errores.field)]]
    if len(t):
        print(f'{sufijo}: {int(t.correct.sum())}/{len(t)} correctos ({100 * t.correct.mean():.1f} %).')
estados = Counter((f, v) for p in predicciones for f, v in (p.get('validation') or {}).items())
display(pd.Series(estados, dtype=int).rename('documentos').reset_index().rename(columns={'level_0': 'validación', 'level_1': 'estado'}))

### 7.1 Dónde funciona bien y dónde no
Exactitud según el dataset (escaneo vs. foto), el tamaño de la imagen y la calidad del OCR (CER).

In [ ]:
cer = pd.read_csv(FINAL / 'evaluation/cer.csv', encoding='utf-8-sig')[['id', 'cer']]
docs = pd.DataFrame([{'id': r['id'], 'mpx': (r['width'] or 0) * (r['height'] or 0) / 1e6} for r in sample])
d = por_doc.merge(docs, on='id').merge(cer, on='id', how='left')
d['tamaño'] = pd.qcut(d.mpx.rank(method='first'), 3, labels=['pequeña', 'mediana', 'grande'])
if d.cer.notna().sum() >= 3:
    d['calidad OCR'] = pd.qcut(d.cer.rank(method='first'), 3, labels=['CER bajo', 'CER medio', 'CER alto'])
filas_g = []
for col in [c for c in ['dataset', 'tamaño', 'calidad OCR'] if c in d]:
    for k, g in d.groupby(col, observed=True):
        filas_g.append({'factor': col, 'grupo': str(k), 'documentos': len(g), 'exactitud %': round(100 * g.ok.sum() / g.n.sum(), 1)})
dondes = pd.DataFrame(filas_g)
dondes.to_csv(EVAL / 'exactitud_por_factor.csv', index=False, encoding='utf-8-sig')
display(dondes)

## 8. Prueba externa independiente (opcional)
La evidencia más honesta de generalización: facturas propias **anotadas a mano por dos personas** y que **no** se usaron para ajustar reglas. Formato del ZIP: `anotaciones.jsonl` + carpeta `imagenes/` (ver [`docs/anotacion_propia.md`](../docs/anotacion_propia.md)). Imágenes y lecturas quedan en una carpeta privada que **no** entra al ZIP de resultados; solo se exportan las métricas.

> Usa documentos **impresos**. Los campos escritos a mano no se reconocen de forma confiable y quedan fuera del alcance del sistema.

In [ ]:
if EXTERNO_ZIP:
    destino = PRIV / 'externo'
    with zipfile.ZipFile(EXTERNO_ZIP) as z:
        for item in z.infolist():
            if not (destino / item.filename).resolve().is_relative_to(destino.resolve()):
                raise ValueError('ZIP con rutas no permitidas')
        z.extractall(destino)
    raiz = next(p.parent for p in destino.rglob('anotaciones.jsonl'))
    externas = load_external_test(raiz)
    vistos = set(REVISADOS.values()) | {r['sha256'] for r in FILAS if r.get('sha256')}
    repetidas = [r['id'] for r in externas if r['sha256'] in vistos]
    if repetidas:
        print('Se excluyen imágenes ya usadas en el desarrollo:', repetidas)
    externas = [r for r in externas if r['sha256'] not in vistos]
    EXT = run(externas, CONFIG_FINAL, PRIV / 'externas', PRIV / 'cache')
    m_ext = metrics(EXT).query("support > 0")
    m_ext.to_csv(EVAL / 'metricas_prueba_externa.csv', index=False, encoding='utf-8-sig')
    display(m_ext[['dataset', 'field', 'support', 'exact_match', 'precision', 'recall', 'f1']].round(3))
    print('Documentos externos evaluados:', len(externas))
else:
    print('Sin ZIP externo: tax_id y document_number quedan sin medición (ver docs/anotacion_propia.md).')

## 9. Leer un documento propio
Carga una factura en imagen o PDF. Verás cada página, los campos detectados, avisos y el texto OCR. Un campo vacío no significa cero. Sin anotaciones no se calcula exactitud.

> Las facturas deben ser **impresas o digitales**. Si los montos, la fecha o el RUC están escritos a mano, la lectura no es confiable.

In [ ]:
PROBAR_DOCUMENTO_EXTERNO = False
DOCUMENTO = ''   # ruta de una imagen/PDF; vacío + PROBAR_DOCUMENTO_EXTERNO=True abre el selector de archivos en Colab
resultados_externos = []
if PROBAR_DOCUMENTO_EXTERNO or DOCUMENTO:
    from PIL import Image, ImageOps
    if not DOCUMENTO:
        from google.colab import files
        subido = files.upload()
        if len(subido) != 1:
            raise ValueError('Selecciona un solo documento')
        DOCUMENTO = next(iter(subido))
    fuente = Path(DOCUMENTO)
    carpeta = PRIV / ('lectura_' + datetime.now(timezone.utc).strftime('%H%M%S'))
    carpeta.mkdir(parents=True, exist_ok=True)
    paginas = render_pdf(fuente, carpeta / 'paginas') if fuente.suffix.lower() == '.pdf' else [fuente]
    for numero, pagina in enumerate(paginas, 1):
        try:
            r = predict(pagina, 'tesseract', CONFIG_FINAL, None); r['revision'] = review_notes(r)
        except Exception as exc:
            r = {'status': 'error', 'error': str(exc), 'fields': {}, 'revision': [str(exc)]}
        r['page'] = numero; resultados_externos.append(r)
        with Image.open(pagina) as im:
            vista = ImageOps.exif_transpose(im).copy(); vista.thumbnail((650, 850)); display(vista)
        display(pd.DataFrame([{'campo': k, 'valor': v if v is not None else 'No detectado'} for k, v in r['fields'].items()]))
        print('Estado:', r['status'], '| cobertura:', extraction_status(r['fields'])['reading_status'], '| avisos:', r['revision'])
    write_json(carpeta / 'resultado.json', resultados_externos)
else:
    print('Activa PROBAR_DOCUMENTO_EXTERNO o indica DOCUMENTO para leer una factura propia.')

In [ ]:
# Opcional: compara solo valores que verificaste en el documento. Formato: {página: {'total': '112.00', ...}}
RESPUESTAS_CORRECTAS = {}
if RESPUESTAS_CORRECTAS and resultados_externos:
    por_pagina = {r['page']: r for r in resultados_externos}
    chequeo = [{'página': p, 'campo': f, 'real': v, 'leído': por_pagina[p]['fields'].get(f),
                'correcto': por_pagina[p]['status'] == 'ok' and normalize(f, por_pagina[p]['fields'].get(f), CONFIG_FINAL['normalization']) == normalize(f, v, CONFIG_FINAL['normalization'])}
               for p, valores in RESPUESTAS_CORRECTAS.items() for f, v in valores.items()]
    chequeo = pd.DataFrame(chequeo); display(chequeo)
    print(f'Exactitud en {len(chequeo)} campos verificados: {chequeo.correcto.mean():.0%} (no generaliza a otros formatos).')

## 10. Guardar resultados
El ZIP reúne las tablas y figuras de la evaluación (en el repositorio, en `results/metrics/evaluacion/` y `results/figures/evaluacion/`). No incluye imágenes, predicciones con texto OCR ni documentos propios.

In [ ]:
resumen_final = {'plan': {k: plan[k] for k in ['revision_codigo', 'seed', 'documents', 'firma_muestra', 'ya_revisados_en_desarrollo']},
                 'configuracion': 'recomendada_04' if RECOMENDADA else 'definitivo.json',
                 'cumplimiento': cumplimiento, 'intervalos': tabla_ic.reset_index().to_dict(orient='records'),
                 'comparacion': comparacion.reset_index().to_dict(orient='records'), 'alertas': dict(alertas)}
write_json(EVAL / 'resumen_evaluacion.json', resumen_final)
archivo = SALIDAS / (EVAL.name + '.zip')
with zipfile.ZipFile(archivo, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in EVAL.rglob('*'):
        if p.is_file() and 'documento_privado' not in p.parts and p.name not in ('predictions.jsonl', 'muestra_test_congelada.jsonl', 'evaluation_manifest.jsonl'):
            z.write(p, p.relative_to(EVAL))
print('ZIP:', archivo)
try:
    from google.colab import files
    files.download(str(archivo))
except ImportError:
    pass

## 11. Conclusiones
- El resultado del proyecto es la exactitud del **modelo final en el test** (sección 4) con su intervalo de confianza (sección 5). Se reporta tal como salió.
- La comparación con M0, M1 y M2 (sección 6) usa las mismas lecturas, así que muestra cuánto aportan las vistas nuevas y el resumen contable.
- Como casi todo el test oficial se revisó durante el desarrollo, la generalización se acredita con la **prueba externa** (sección 8); `tax_id` y `document_number` solo se pueden medir ahí.
- El resumen de resultados está en el [README](../README.md#4-resultados) y en [`results/`](../results/).